In [1]:
from physics.simulation import mcfm, msq
from physics.analysis import zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "text.usetex": True,
    "pgf.texsystem": "lualatex",
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [2]:
with open('data/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : xsec['gg4l_sbi'],
    msq.Component.SIG : xsec['gg4l_sig'],
    msq.Component.INT : xsec['gg4l_int'],
    msq.Component.BKG : xsec['gg4l_bkg']
}

filenames = {
    msq.Component.SBI : 'data/ggZZ2e2m_sbi/events.csv',
    msq.Component.SIG : 'data/ggZZ2e2m_sig/events.csv',
    msq.Component.INT : 'data/ggZZ2e2m_int/events.csv',
    msq.Component.BKG : 'data/ggZZ2e2m_bkg/events.csv'
}

component_names = {
    msq.Component.SBI : 'SBI',
    msq.Component.SIG : 'SIG',
    msq.Component.INT : 'INT',
    msq.Component.BKG : 'BKG'
}

In [3]:
events_sbi = mcfm.from_csv(cross_section=xsec[msq.Component.SBI], file_path=filenames[msq.Component.SBI], n_rows=10000)
events_sbi = zz4l.analyze(events_sbi).shuffle()
events_sig = events_sbi.reweight(msq.Component.SBI, msq.Component.SIG)
events_int = events_sbi.reweight(msq.Component.SBI, msq.Component.INT)

In [4]:
c6_vals = np.linspace(-25, 25, 101)
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

c6_pts_sig = np.array([-10,-5,0, 5,10])
c6_pts_int = np.array([-10,   0,   10])
c6_pts_sbi = np.array([-10,-5,0, 5,10])

mod_sig_c6 = c6.Modifier(baseline=msq.Component.SIG, events=events_sig, c6_points=c6_pts_sig)
mod_int_c6 = c6.Modifier(baseline=msq.Component.INT, events=events_int, c6_points=c6_pts_int)
mod_sbi_c6 = c6.Modifier(baseline=msq.Component.SBI, events=events_sbi, c6_points=c6_pts_sbi)

wt_sig_c6, prob_sig_c6 = mod_sig_c6.modify(c6_vals)
wt_int_c6, prob_int_c6 = mod_int_c6.modify(c6_vals)
wt_sbi_c6, prob_sbi_c6 = mod_sbi_c6.modify(c6_vals)

In [5]:
sig_bsm_over_sm = wt_sig_c6 / events_sig.weights.to_numpy()[:,np.newaxis]
int_bsm_over_sm = wt_int_c6 / events_int.weights.to_numpy()[:,np.newaxis]
sbi_bsm_over_sm = wt_sbi_c6 / events_sbi.weights.to_numpy()[:,np.newaxis]

In [12]:
event_index = 123

In [18]:
fig, (ax_sig, ax_int, ax_sbi) = plt.subplots(3,1, gridspec_kw={'height_ratios': [1,1,1]}, figsize=(5,6), sharex=True)

# ax_sig.plot(c6_vals, sig_bsm_over_sm[90], linestyle='--')
# ax_int.plot(c6_vals, int_bsm_over_sm[90], linestyle='--')
# ax_sbi.plot(c6_vals, sbi_bsm_over_sm[90], linestyle='--')

msq_sig_c6 = wt_sig_c6 / events_sig.weights.to_numpy()[:,np.newaxis] #* events_sig.components['msq_sig_sm'].to_numpy()[:,np.newaxis]
msq_int_c6 = wt_int_c6 / events_int.weights.to_numpy()[:,np.newaxis] #* events_int.components['msq_int_sm'].to_numpy()[:,np.newaxis]
msq_sbi_c6 = wt_sbi_c6 / events_sbi.weights.to_numpy()[:,np.newaxis] #* events_sbi.components['msq_sbi_sm'].to_numpy()[:,np.newaxis]

msq_sig_sm = events_sig.components['msq_sig_sm'].to_numpy()[event_index]
msq_int_sm = events_int.components['msq_int_sm'].to_numpy()[event_index]
msq_sbi_sm = events_sbi.components['msq_sbi_sm'].to_numpy()[event_index]

c6_pts_all = []
msq_sig_pts = []
msq_int_pts = []
msq_sbi_pts = []
for c6_val, msq_name in mcfm.mcfm_component_c6[msq.Component.SBI].items():
    c6_pts_all.append(c6_val)
    msq_sbi_pts.append(events_sbi.components[msq_name][event_index])
for msq_name in mcfm.mcfm_component_c6[msq.Component.SIG].values():
    msq_sig_pts.append(events_sig.components[msq_name][event_index])
for msq_name in mcfm.mcfm_component_c6[msq.Component.INT].values():
    msq_int_pts.append(events_int.components[msq_name][event_index])
msq_sig_pts = np.array(msq_sig_pts)
msq_int_pts = np.array(msq_int_pts)
msq_sbi_pts = np.array(msq_sbi_pts)

ax_sig.plot(c6_vals, msq_sig_c6[event_index], linestyle='--', color='blue')
ax_int.plot(c6_vals, msq_int_c6[event_index], linestyle='--', color='blue')
ax_sbi.plot(c6_vals, msq_sbi_c6[event_index], linestyle='--', color='blue')

ax_sig.scatter(c6_pts_all, msq_sig_pts / msq_sig_sm, color='black')
ax_int.scatter(c6_pts_all, msq_int_pts / msq_int_sm, color='black')
ax_sbi.scatter(c6_pts_all, msq_sbi_pts / msq_sbi_sm, color='black')

ax_sig.scatter(c6_pts_sig, msq_sig_pts[np.nonzero(np.isin(c6_pts_all, c6_pts_sig))[0]] / msq_sig_sm, color='blue')
ax_int.scatter(c6_pts_int, msq_int_pts[np.nonzero(np.isin(c6_pts_all, c6_pts_int))[0]] / msq_int_sm, color='blue')
ax_sbi.scatter(c6_pts_sbi, msq_sbi_pts[np.nonzero(np.isin(c6_pts_all, c6_pts_sbi))[0]] / msq_sbi_sm, color='blue')

ax_sig.text(0.04 ,0.96, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=ax_sig.transAxes, ha='left', va='top', fontsize=12)
ax_sbi.text(0.04 ,0.04, '$gg \\rightarrow ZZ$', transform=ax_sbi.transAxes, ha='left', va='bottom', fontsize=12)

ax_sig.set_ylabel('$|\\mathcal{M}_{\\mathrm{s}}|^2 / \\mathrm{SM}$', fontsize=15)
ax_int.set_ylabel('$2\\mathrm{Re}(\\mathcal{M}^{\\dag}_{\\mathrm{s}} \\mathcal{M}_{\\mathrm{b}}) / \\mathrm{SM}$', fontsize=15)
ax_sbi.set_ylabel('$|\\mathcal{M}_{\\mathrm{s}} + \\mathcal{M}_{\\mathrm{b}}|^2 / \\mathrm{SM}$', fontsize=15)

# ax_sig.set_ylim( 0.0, 1.5)
# ax_int.set_ylim(-1.0, 2.0)
# ax_sbi.set_ylim( 0.0, 1.5)

# ax_sig.yaxis.set_ticks([])
# ax_int.yaxis.set_ticks([])
# ax_sbi.yaxis.set_ticks([])

ax_sbi.set_xlabel('$c_6$', fontsize=15)
ax_sbi.set_xlim(-26,26)
ax_sbi.xaxis.set_ticks([-25,-20,-15,-10,-5,0,5,10,15,20,25])

ax_sbi.tick_params(top=True, labeltop=False)
ax_int.tick_params(top=True, labeltop=False)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

plt.savefig('msq_bsm_over_sm.pdf')
plt.close()